# C3 Ensemble clean Kaggle pipeline

This notebook is a thin, auditable launcher for the modular implementation in `src/c3_clean`. It never modifies legacy notebooks or raw CSV files. P0 must pass before P1-P3 can execute. All outputs are written under `/kaggle/working/c3_clean_artifacts/`.

## Three-person Kaggle workflow

Use the **same notebook** in three Kaggle sessions. Set `JOB_MODE = 'seed_worker'` and assign one person each to seed `42`, `1`, or `7`; each worker publishes its `c3_clean_artifacts` output as a private Kaggle dataset. A fourth, lightweight finalization session attaches those three worker datasets, sets `JOB_MODE = 'ensemble_finalizer'`, and creates the C3 Ensemble without retraining. The runtime cell contains all toggles.

Training output is redirected to per-job log files by default, keeping Papermill notebook output small. The finalizer imports only the arrays and ID-order files by default; enable checkpoint import only for a final archival/package run with enough disk space.

In [ ]:
from pathlib import Path
import contextlib, os, sys, subprocess, yaml, shutil

REPOSITORY_URL = 'https://github.com/FlynnBui399/vietnamese-emoji-emotion-recognition.git'
REPOSITORY_REF = 'main'
CLONE_IF_MISSING = True
CLONE_DESTINATION = Path('/kaggle/working/vietnamese-emoji-emotion-recognition')

def find_repository_root():
    direct = [Path.cwd(), Path('/kaggle/working')]
    for root in direct:
        if (root / 'configs' / 'c3_clean.yaml').is_file():
            return root.resolve()
    for search_root in (Path('/kaggle/input'), Path('/kaggle/working')):
        if search_root.is_dir():
            for match in search_root.rglob('configs/c3_clean.yaml'):
                return match.parent.parent.resolve()
    return None

REPO_ROOT = find_repository_root()
if REPO_ROOT is None and CLONE_IF_MISSING:
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Automatic cloning is intended for Kaggle')
    subprocess.check_call([
        'git', 'clone', '--depth', '1', '--branch', REPOSITORY_REF,
        REPOSITORY_URL, str(CLONE_DESTINATION),
    ])
    REPO_ROOT = CLONE_DESTINATION.resolve()
if REPO_ROOT is None:
    raise FileNotFoundError('Attach the updated repository or enable CLONE_IF_MISSING')
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print('Repository:', REPO_ROOT)

In [ ]:
# Job toggle: 'audit', 'seed_worker', 'ensemble_finalizer', or 'full'.
# For three C3 workers, make three notebook copies and set ASSIGNED_SEED to 42, 1, and 7.
JOB_MODE = 'seed_worker'
WORK_PRIORITY = 'P1'  # A seed worker normally runs P1; use P2 only for distributed ablations.
ASSIGNED_SEED = 42    # Exactly one of 42, 1, or 7 when JOB_MODE == 'seed_worker'.

# Attach each published worker output in the finalizer and list its c3_clean_artifacts directory.
# Example: ['/kaggle/input/c3-seed42/c3_clean_artifacts', ...]
ARTIFACT_IMPORT_SOURCES = []
IMPORT_CHECKPOINTS = False  # False imports only arrays/IDs/metrics: far less disk for ensemble evaluation.
CREATE_FINAL_ZIP = False    # Enable only in an archival finalizer session with enough free disk.
QUIET_EXECUTION = True       # Redirect training/warning output to logs, avoiding Papermill notebook bloat.

# Runtime values are explicit and are recorded in every resolved seed config.
INSTALL_DEPENDENCIES = False
DATA_DIR_OVERRIDE = None       # Example: '/kaggle/input/c3-clean-inputs/data/vigoemotions'
EMOJI2VEC_OVERRIDE = None      # Example: '/kaggle/input/c3-clean-inputs/data/emoji2vec.bin'
VISOBERT_MODEL_OVERRIDE = None # Set to an attached HF snapshot when Kaggle internet is off.
NUM_GPUS = 1          # Set exactly 2 only in a Kaggle two-GPU session.
PRECISION = 'fp32'    # One of: fp32, fp16, bf16. No automatic fallback.
BATCH_SIZE = 32
GRADIENT_ACCUMULATION = 1
MAX_LENGTH = 128

RUN_EXPERIMENTS = [
    'A0_controlled_text_BCE',
    'A1_controlled_text_ASL',
    'A2_controlled_ASL_Emoji',
    'A3_controlled_ASL_Emoji_CB',
]
OPTIONAL_EXPERIMENTS = [
    'Emoji-random-control',
    'Emoji-shuffle-control',
    'Emoji-zero-control',
    'C3-RDrop',
    'C3-extended-matched',
]

if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')])

if JOB_MODE not in {'audit', 'seed_worker', 'ensemble_finalizer', 'full'}:
    raise ValueError(f'Unknown JOB_MODE: {JOB_MODE}')
if WORK_PRIORITY not in {'P1', 'P2', 'P3'}:
    raise ValueError('WORK_PRIORITY must be P1, P2, or P3')
if JOB_MODE == 'seed_worker' and ASSIGNED_SEED not in {42, 1, 7}:
    raise ValueError('ASSIGNED_SEED must be exactly 42, 1, or 7')

working_artifacts = Path('/kaggle/working/c3_clean_artifacts')
MINIMAL_HANDOFF_NAMES = {
    'val_probs.npy', 'val_targets.npy', 'test_probs.npy', 'test_targets.npy',
    'val_ids.json', 'test_ids.json', 'thresholds.json', 'metrics_exact.json',
    'training_history.csv', 'config.json', 'best_epoch.json',
}

def artifact_root(source):
    source = Path(source)
    nested = source / 'c3_clean_artifacts'
    return nested if nested.is_dir() else source

def import_worker_artifacts(source):
    source_root = artifact_root(source)
    if not source_root.is_dir():
        raise FileNotFoundError(f'Worker artifact directory not found: {source_root}')
    copied = 0
    for item in source_root.rglob('*'):
        if not item.is_file():
            continue
        keep = IMPORT_CHECKPOINTS or item.name in MINIMAL_HANDOFF_NAMES
        if not keep:
            continue
        destination = working_artifacts / item.relative_to(source_root)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, destination)
        copied += 1
    print(f'Imported {copied} artifact files from: {source_root}')

for artifact_source in ARTIFACT_IMPORT_SOURCES:
    import_worker_artifacts(artifact_source)

base_config_path = REPO_ROOT / 'configs' / 'c3_clean.yaml'
config = yaml.safe_load(base_config_path.read_text(encoding='utf-8'))
config['paths']['repository_root'] = str(REPO_ROOT)
if DATA_DIR_OVERRIDE is not None:
    config['paths']['data_dir_override'] = DATA_DIR_OVERRIDE
if EMOJI2VEC_OVERRIDE is not None:
    config['paths']['emoji2vec'] = EMOJI2VEC_OVERRIDE
if VISOBERT_MODEL_OVERRIDE is not None:
    config['model']['model_name'] = VISOBERT_MODEL_OVERRIDE
config['runtime']['num_gpus'] = NUM_GPUS
config['runtime']['precision'] = PRECISION
config['training']['batch_size'] = BATCH_SIZE
config['training']['gradient_accumulation'] = GRADIENT_ACCUMULATION
config['model']['max_length'] = MAX_LENGTH
runtime_config_path = Path('/kaggle/working/c3_clean_runtime.yaml')
runtime_config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')

def run_pipeline(label, arguments):
    log_path = working_artifacts / 'logs' / f'{label}.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    if QUIET_EXECUTION:
        with log_path.open('a', encoding='utf-8') as log_handle, \
             contextlib.redirect_stdout(log_handle), contextlib.redirect_stderr(log_handle):
            from src.c3_clean.run_experiments import main
            status = main(arguments)
    else:
        from src.c3_clean.run_experiments import main
        status = main(arguments)
    print(f'{label}: status={status}; log={log_path}')
    return status

print('Runtime config:', runtime_config_path)
print(f'Job: {JOB_MODE}; priority: {WORK_PRIORITY}; seed: {ASSIGNED_SEED}')

## P0 - source audit and artifact-only reconstruction

This cell hashes and audits the raw splits, inventories attached legacy bundles, verifies available checkpoints, averages the three validation/test probability arrays, and fits new thresholds only on averaged validation probabilities. A data discrepancy exits with status 2 after writing the discrepancy report.

In [ ]:
if JOB_MODE in {'audit', 'seed_worker', 'ensemble_finalizer', 'full'}:
    P0_STATUS = run_pipeline('P0_audit', ['--config', str(runtime_config_path), '--priority', 'P0'])
    if P0_STATUS != 0:
        raise RuntimeError('P0 stopped. Inspect /kaggle/working/c3_clean_artifacts/data_discrepancies.md before continuing.')

## P1 - canonical C3 seeds and ensemble

For distributed C3 training, each person runs this cell once with `JOB_MODE = 'seed_worker'` and a different `ASSIGNED_SEED`. Publish each worker's `c3_clean_artifacts` directory as a private Kaggle dataset. In the finalizer, attach all three outputs, list them in `ARTIFACT_IMPORT_SOURCES`, set `JOB_MODE = 'ensemble_finalizer'`, and run this cell again. Completed seed artifacts are never retrained.

In [ ]:
if WORK_PRIORITY == 'P1' and JOB_MODE == 'seed_worker':
    P1_STATUS = run_pipeline('P1_seed' + str(ASSIGNED_SEED), [
        '--config', str(runtime_config_path), '--priority', 'P1',
        '--seeds', str(ASSIGNED_SEED),
        '--run-experiments', 'A3_controlled_ASL_Emoji_CB',
    ])
elif WORK_PRIORITY == 'P1' and JOB_MODE == 'ensemble_finalizer':
    P1_STATUS = run_pipeline('P1_ensemble_finalizer', [
        '--config', str(runtime_config_path), '--priority', 'P1', '--assemble-only',
    ])
elif WORK_PRIORITY == 'P1' and JOB_MODE == 'full':
    P1_STATUS = run_pipeline('P1_full', [
        '--config', str(runtime_config_path), '--priority', 'P1',
        '--run-experiments', 'A3_controlled_ASL_Emoji_CB',
    ])
else:
    print('P1 skipped by the current job toggle.')
    P1_STATUS = 0
if P1_STATUS != 0:
    raise RuntimeError('P1 failed; inspect the corresponding logs/P1_*.log file.')

## P2 - controlled A0-A3 ablation and matched ensembles

This supports the same worker/finalizer toggle. A distributed P2 worker trains its assigned seed for every selected ablation. The finalizer requires all three seed artifacts for both A0 and C3 before it calculates matched ensemble statistics.

In [ ]:
if WORK_PRIORITY == 'P2' and JOB_MODE == 'seed_worker':
    P2_STATUS = run_pipeline('P2_seed' + str(ASSIGNED_SEED), [
        '--config', str(runtime_config_path), '--priority', 'P2',
        '--seeds', str(ASSIGNED_SEED), '--run-experiments', *RUN_EXPERIMENTS,
    ])
elif WORK_PRIORITY == 'P2' and JOB_MODE == 'ensemble_finalizer':
    P2_STATUS = run_pipeline('P2_ensemble_finalizer', [
        '--config', str(runtime_config_path), '--priority', 'P2', '--assemble-only',
    ])
elif WORK_PRIORITY == 'P2' and JOB_MODE == 'full':
    P2_STATUS = run_pipeline('P2_full', [
        '--config', str(runtime_config_path), '--priority', 'P2',
        '--run-experiments', *RUN_EXPERIMENTS,
    ])
else:
    print('P2 skipped by the current job toggle.')
    P2_STATUS = 0
if P2_STATUS != 0:
    raise RuntimeError('P2 failed; inspect the corresponding logs/P2_*.log file.')

## P3 - optional diagnostics

Run only after P0-P2 are complete. These results remain separate from the canonical C3 headline result.

In [ ]:
if WORK_PRIORITY == 'P3' and JOB_MODE == 'seed_worker':
    P3_STATUS = run_pipeline('P3_seed' + str(ASSIGNED_SEED), [
        '--config', str(runtime_config_path), '--priority', 'P3',
        '--seeds', str(ASSIGNED_SEED), '--run-experiments', *OPTIONAL_EXPERIMENTS,
    ])
elif WORK_PRIORITY == 'P3' and JOB_MODE == 'full':
    P3_STATUS = run_pipeline('P3_full', [
        '--config', str(runtime_config_path), '--priority', 'P3',
        '--run-experiments', *OPTIONAL_EXPERIMENTS,
    ])
else:
    print('P3 skipped by the current job toggle.')
    P3_STATUS = 0
if P3_STATUS != 0:
    raise RuntimeError('P3 failed; inspect the corresponding logs/P3_*.log file.')

## Final package

Create the required ZIP only after the selected priorities have completed.

In [ ]:
if CREATE_FINAL_ZIP:
    from src.c3_clean.run_experiments import package_kaggle_artifacts
    archive = package_kaggle_artifacts(Path('/kaggle/working/c3_clean_artifacts'))
    print('Created final archive:', archive)
else:
    print('Final ZIP skipped. Set CREATE_FINAL_ZIP = True only for a final archival run with enough disk.')